In [ ]:
# Setup.
import os
import sys
from pathlib import Path

for _candidate in [".", "..", "../.."]:  # noqa: B007
    _p = os.path.abspath(_candidate)
    if os.path.isdir(os.path.join(_p, "src")):
        sys.path.insert(0, _p)
        break

import matplotlib.pyplot as plt  # noqa: E402
import mne  # noqa: E402
from matplotlib.cm import ScalarMappable  # noqa: E402
from matplotlib.colors import Normalize  # noqa: E402
from matplotlib.lines import Line2D  # noqa: E402
from matplotlib.patches import Rectangle  # noqa: E402
from matplotlib.transforms import blended_transform_factory  # noqa: E402
import numpy as np  # noqa: E402
import pandas as pd  # noqa: E402

from src.analysis import iva_quality  # noqa: E402
from src.definitions.constants import AssrEpoch, ProjectPaths  # noqa: E402
from src.definitions.fields import (  # noqa: E402
    ConditionVariants,
    CoordinateSystems,
    ExperimentNames,
    IvaVariants,
    MusicTypeVariants,
    SpectrumTypeVariants,
)
from src.io.iva_store import load_iva_components  # noqa: E402
from src.io.loading import (  # noqa: E402
    assr_electrode_mask,
    load_snr_courses,
    load_snr_values,
)
from src.models.synchrony import candidate_intervals, score_track  # noqa: E402
from src.visualization.assr_snr_plots import (  # noqa: E402
    CONDITION_COLORS,
    DIVERGING_CMAP,
    plot_condition_tf_grid,
    plot_courses,
)
from src.visualization.corrca_plots import (  # noqa: E402
    COMPONENT_PREFIX,
    plot_shared_topomaps,
)

# The imported scripts switch matplotlib to the headless backend; switch back.
%matplotlib inline

print("Setup complete.")

# Conference poster figures

Illustrative figures for the conference poster. Nothing here reads data or feeds the
pipeline; each section draws one poster graphic and saves it under
`plots/conference/<figure>/`.

One colour-blind-safe palette throughout: the Okabe-Ito blue `#0072B2` and vermillion
`#D55E00` (`CONDITION_COLORS` in `src/visualization/assr_snr_plots.py`, Placebo and
Psilocybin in every condition figure), near-black and greys for everything else, and
`DIVERGING_CMAP` (the same two hues through near-white, equally dark at both ends) for
the TF maps.

## Schematic EEG traces

A stack of synthetic EEG traces with only the electrode names: E1-E3, a vertical
ellipsis standing in for the omitted channels, then E254-E256. No axes, ticks or
scales.

The signal is synthetic: a 1/f background, intermittent alpha bursts (~10 Hz) and a
weak theta rhythm, low-passed at `LOWPASS_HZ` so the lines stay smooth. Each trace mixes a component shared by its block (E1-E3 or
E254-E256) with its own noise, so neighbouring electrodes look alike, as they do
in real recordings. Each variant is one random seed plus a few shape parameters, and is saved twice:
with the electrode names and without them (`_nolabels`).

In [ ]:
TOP_ELECTRODES = ["E1", "E2", "E3"]
BOTTOM_ELECTRODES = ["E254", "E255", "E256"]

SFREQ = 250.0  # Hz
LOWPASS_HZ = 20.0  # lower it for smoother traces
LINE_COLOR = "#1f2937"
LINE_WIDTH = 1.0
TRACE_SPACING = 3.5  # vertical distance between traces, in z-score units
TRACE_SCALE = 0.8  # trace amplitude multiplier, lowered per figure if traces touch
TRACE_MIN_GAP = 0.15  # smallest gap between neighbouring traces, x spacing
# Gaps are measured over a sliding window of this fraction of the trace, so a steep
# peak cannot slip past a neighbour's trough at an adjacent time point.
TRACE_GAP_WINDOW = 0.02
DOT_SIZE = 5  # ellipsis dots, in points^2
LABEL_FONTSIZE = 11
FIGSIZE = (8.0, 4.5)  # inches
TRANSPARENT = True  # transparent background, for placing on a coloured poster

# One entry per variant: seed, duration in seconds, alpha strength relative to the
# 1/f background, and how much of each trace is shared with its neighbours.
VARIANTS = {
    "variant_1": {"seed": 1, "duration": 4.0, "alpha": 1.0, "shared": 0.5},
    "variant_2": {"seed": 7, "duration": 3.0, "alpha": 1.4, "shared": 0.6},
    "variant_3": {"seed": 23, "duration": 5.0, "alpha": 0.7, "shared": 0.4},
}

OUTPUT_DIR = ProjectPaths.analysis_plots_dir("conference", "eeg_traces")

In [ ]:
def pink_noise(rng: np.random.Generator, n: int, exponent: float = 1.0) -> np.ndarray:
    """Noise with a 1/f^exponent power spectrum, zero mean and unit variance."""
    spectrum = rng.standard_normal(n // 2 + 1) + 1j * rng.standard_normal(n // 2 + 1)
    freqs = np.fft.rfftfreq(n)
    freqs[0] = freqs[1]
    signal = np.fft.irfft(spectrum / freqs ** (exponent / 2), n)
    return (signal - signal.mean()) / signal.std()


def lowpass(signal: np.ndarray, sfreq: float, cutoff: float) -> np.ndarray:
    """Zero-phase low-pass with a smooth (Gaussian) roll-off, so it does not ring."""
    freqs = np.fft.rfftfreq(signal.shape[-1], d=1 / sfreq)
    gain = np.exp(-0.5 * (freqs / cutoff) ** 2)
    return np.fft.irfft(np.fft.rfft(signal, axis=-1) * gain, signal.shape[-1], axis=-1)


def rhythm(
    rng: np.random.Generator,
    times: np.ndarray,
    freq: float,
    burst_rate: float,
) -> np.ndarray:
    """A jittered oscillation under a slowly varying, bursty envelope."""
    jitter = 0.5 * pink_noise(rng, times.size, exponent=2.0)
    phase = 2 * np.pi * np.cumsum(freq + jitter) / SFREQ
    envelope = np.clip(pink_noise(rng, times.size, exponent=2.0) + 0.3, 0, None)
    envelope *= 1 + np.sin(2 * np.pi * burst_rate * times + rng.uniform(0, 2 * np.pi))
    return envelope * np.sin(phase)


def synthetic_eeg(
    rng: np.random.Generator,
    n_channels: int,
    duration: float,
    alpha: float,
    shared: float,
    lowpass_hz: float = LOWPASS_HZ,
    roughness: float = 0.0,
) -> np.ndarray:
    """``(n_channels, n_times)`` synthetic EEG, each channel z-scored.

    ``lowpass_hz`` and ``roughness`` (broadband noise, relative to the 1/f background)
    set how coarse the traces look.
    """
    times = np.arange(int(duration * SFREQ)) / SFREQ

    def source() -> np.ndarray:
        signal = (
            pink_noise(rng, times.size, exponent=1.8)
            + alpha * rhythm(rng, times, freq=10.0, burst_rate=0.4)
            + 0.4 * rhythm(rng, times, freq=6.0, burst_rate=0.2)
        )
        if roughness:
            signal = signal + roughness * rng.standard_normal(times.size)
        return signal

    common = source()
    traces = np.stack(
        [shared * common + (1 - shared) * source() for _ in range(n_channels)]
    )
    traces = lowpass(traces, SFREQ, lowpass_hz)
    return (traces - traces.mean(axis=1, keepdims=True)) / traces.std(
        axis=1, keepdims=True
    )

In [ ]:
def _non_overlapping_scale(blocks: list[np.ndarray], spacing: float) -> float:
    """``TRACE_SCALE``, shrunk just enough that neighbouring traces never meet.

    Within each window, the upper trace's lowest point ``a`` and the lower trace's
    highest point ``b`` are ``spacing + s * (a - b)`` apart at scale ``s``, so the gap
    stays above ``TRACE_MIN_GAP * spacing`` while
    ``s <= (1 - TRACE_MIN_GAP) * spacing / max(b - a)``.
    """
    from numpy.lib.stride_tricks import sliding_window_view

    window = max(1, int(TRACE_GAP_WINDOW * blocks[0].shape[1]))
    scale = TRACE_SCALE
    for block in blocks:
        for upper, lower in zip(block[:-1], block[1:]):
            lowest = sliding_window_view(upper, window).min(axis=-1)
            highest = sliding_window_view(lower, window).max(axis=-1)
            approach = (highest - lowest).max()
            if approach > 0:
                scale = min(scale, (1 - TRACE_MIN_GAP) * spacing / approach)
    return scale


def plot_eeg_traces(
    top: np.ndarray,
    bottom: np.ndarray,
    show_labels: bool = True,
    save_path: str | os.PathLike | None = None,
) -> plt.Figure:
    """Stacked traces, optionally labelled by electrode name, with an ellipsis
    between the two blocks."""
    spacing = TRACE_SPACING
    gap = 1.6  # the ellipsis row is this many spacings tall
    labels = TOP_ELECTRODES + BOTTOM_ELECTRODES
    offsets = [-i * spacing for i in range(len(TOP_ELECTRODES))]
    dots_y = offsets[-1] - (gap + 1) * spacing / 2
    first_bottom = offsets[-1] - (gap + 1) * spacing
    offsets += [first_bottom - i * spacing for i in range(len(BOTTOM_ELECTRODES))]

    scale = _non_overlapping_scale([top, bottom], spacing)
    n_times = top.shape[1]
    x = np.linspace(0.0, 1.0, n_times)
    fig, ax = plt.subplots(figsize=FIGSIZE)
    for trace, offset, label in zip(np.vstack([top, bottom]), offsets, labels):
        ax.plot(x, scale * trace + offset, color=LINE_COLOR, linewidth=LINE_WIDTH)
        if not show_labels:
            continue
        ax.text(
            -0.03,
            offset,
            label,
            ha="right",
            va="center",
            fontsize=LABEL_FONTSIZE,
            color=LINE_COLOR,
        )

    # Vertical ellipsis across the middle of the traces, and under the labels.
    dot_ys = dots_y + np.array([-1, 0, 1]) * spacing * 0.2
    for dot_x in (-0.055, 0.5) if show_labels else (0.5,):
        ax.scatter(
            np.full(3, dot_x), dot_ys, s=DOT_SIZE, color=LINE_COLOR, clip_on=False
        )

    ax.set_xlim(0, 1)
    ax.set_ylim(offsets[-1] - spacing, offsets[0] + spacing)
    ax.axis("off")
    fig.tight_layout()
    if save_path is not None:
        fig.savefig(save_path, dpi=300, bbox_inches="tight", transparent=TRANSPARENT)
    return fig

In [ ]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

for name, params in VARIANTS.items():
    rng = np.random.default_rng(params["seed"])
    kwargs = {k: params[k] for k in ("duration", "alpha", "shared")}
    top = synthetic_eeg(rng, len(TOP_ELECTRODES), **kwargs)
    bottom = synthetic_eeg(rng, len(BOTTOM_ELECTRODES), **kwargs)

    for show_labels, suffix in ((True, ""), (False, "_nolabels")):
        stem = OUTPUT_DIR / f"{name}{suffix}"
        fig = plot_eeg_traces(
            top, bottom, show_labels=show_labels, save_path=f"{stem}.png"
        )
        fig.savefig(f"{stem}.svg", bbox_inches="tight", transparent=TRANSPARENT)
        plt.show()
    print(f"{name}: saved to {OUTPUT_DIR}")

## Schematic audio waveform

A bare audio waveform for the stimulus side of the poster, drawn as discrete
symmetric bars (one per time column, height = the column's RMS): no axes, no
labels. The sound is a synthetic sustained chord; what the bars show is its loudness
envelope, a slow random swell with faster fluctuations on top. Each variant is one
random seed.

In [ ]:
AUDIO_SFREQ = 8000.0  # Hz; enough to draw a dense waveform, not for listening
AUDIO_DURATION = 6.0  # seconds
AUDIO_COLOR = LINE_COLOR
AUDIO_BINS = 180  # number of bars
AUDIO_BAR_FILL = 0.7  # fraction of each column the bar fills; the rest is gap
AUDIO_MIN_HEIGHT = 0.04  # so near-silent columns still show a short bar
AUDIO_FIGSIZE = (8.0, 1.8)  # inches
# Loudness envelope around a mean level of 1: how far the slow swell and the faster
# fluctuations reach. Higher is more variable.
AUDIO_SWELL = 0.35
AUDIO_FLUTTER = 0.25
AUDIO_SEEDS = {"variant_1": 3, "variant_2": 8, "variant_3": 21}

AUDIO_OUTPUT_DIR = ProjectPaths.analysis_plots_dir("conference", "audio_waveform")

In [ ]:
def _unit(signal: np.ndarray) -> np.ndarray:
    return signal / np.abs(signal).max()


def classic_audio(rng: np.random.Generator, times: np.ndarray) -> np.ndarray:
    """Sustained chord with vibrato under a randomly swelling loudness envelope."""
    chord = [220.0, 277.2, 329.6, 440.0]  # A major
    signal = np.zeros_like(times)
    for freq in chord:
        vibrato = 1 + 0.004 * np.sin(2 * np.pi * rng.uniform(4, 6) * times)
        for harmonic, weight in ((1, 1.0), (2, 0.4), (3, 0.2)):
            signal += weight * np.sin(
                2 * np.pi * harmonic * freq * np.cumsum(vibrato) / AUDIO_SFREQ
            )
    signal = _unit(signal) + 0.15 * rng.standard_normal(times.size)

    def smooth_noise(cutoff: float) -> np.ndarray:
        noise = lowpass(rng.standard_normal(times.size), AUDIO_SFREQ, cutoff)
        return noise / noise.std()

    envelope = 1 + AUDIO_SWELL * smooth_noise(0.5) + AUDIO_FLUTTER * smooth_noise(3)
    envelope = np.clip(envelope, 0.1, None)
    return envelope * signal


def plot_audio_waveform(
    signal: np.ndarray,
    save_path: str | os.PathLike | None = None,
) -> plt.Figure:
    """A bare waveform drawn as discrete bars, one per column, with no axes.

    Each bar is the column's RMS, lightly smoothed across neighbours: the RMS follows
    loudness without the spikes a per-column peak shows.
    """
    columns = signal[: signal.size // AUDIO_BINS * AUDIO_BINS]
    rms = np.sqrt((columns.reshape(AUDIO_BINS, -1) ** 2).mean(axis=1))
    rms = np.convolve(rms, [0.25, 0.5, 0.25], mode="same")
    height = np.maximum(rms / rms.max(), AUDIO_MIN_HEIGHT)
    x = (np.arange(AUDIO_BINS) + 0.5) / AUDIO_BINS
    fig, ax = plt.subplots(figsize=AUDIO_FIGSIZE)
    ax.bar(
        x,
        2 * height,
        bottom=-height,
        width=AUDIO_BAR_FILL / AUDIO_BINS,
        color=AUDIO_COLOR,
        linewidth=0,
    )
    ax.set_xlim(0, 1)
    ax.set_ylim(-1.05, 1.05)
    ax.axis("off")
    fig.tight_layout()
    if save_path is not None:
        fig.savefig(save_path, dpi=300, bbox_inches="tight", transparent=TRANSPARENT)
    return fig

In [ ]:
AUDIO_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

audio_times = np.arange(int(AUDIO_DURATION * AUDIO_SFREQ)) / AUDIO_SFREQ
for name, seed in AUDIO_SEEDS.items():
    audio = classic_audio(np.random.default_rng(seed), audio_times)
    fig = plot_audio_waveform(audio, save_path=AUDIO_OUTPUT_DIR / f"{name}.png")
    fig.savefig(
        AUDIO_OUTPUT_DIR / f"{name}.svg", bbox_inches="tight", transparent=TRANSPARENT
    )
    plt.show()
    print(f"{name}: saved to {AUDIO_OUTPUT_DIR}")

## Synchrony time course with threshold

Group synchrony over time, with the significance threshold drawn as a grey dashed
line and everything below it in a light grey band. Only brief, narrow peaks
cross it (the run prints the share of windows above). It is computed exactly as stage 08 scores a track
(`score_track` in [`src/models/synchrony.py`](../../src/models/synchrony.py)): the
mean pairwise cosine similarity between subjects' embeddings in each window, against
a threshold set at a quantile of circularly time-shifted surrogates.

Only the embeddings are synthetic: each subject's embedding is its own slowly drifting
vector plus a vector shared by all subjects, whose weight rises during a few
"synchrony episodes". Outside the episodes the subjects only agree by chance, so
the curve hovers under the threshold.

In [ ]:
SYNC_N_SUBJECTS = 8  # fewer subjects give a noisier curve
SYNC_N_WINDOWS = 400
SYNC_DIM = 16
SYNC_N_EPISODES = 9  # how many brief peaks of shared signal
SYNC_EPISODE_WIDTH = (0.006, 0.012)  # range, as a fraction of the track
SYNC_EPISODE_STRENGTH = 1.1  # weight of the shared vector at an episode's peak
SYNC_SMOOTHING = 0.1  # drift cutoff in cycles/window; higher is coarser
SYNC_NULL_QUANTILE = 0.99
SYNC_N_NULL = 200

SYNC_COLOR = "#0072B2"  # Okabe-Ito blue, as Placebo in CONDITION_COLORS
SYNC_THRESHOLD_COLOR = "#6b7280"
SYNC_BELOW_COLOR = "#9ca3af"  # band under the threshold
SYNC_BELOW_ALPHA = 0.25
SYNC_LINE_WIDTH = 2.0
SYNC_FIGSIZE = (8.0, 2.5)  # inches
SYNC_SEEDS = {"variant_1": 4, "variant_2": 11, "variant_3": 30}

SYNC_OUTPUT_DIR = ProjectPaths.analysis_plots_dir("conference", "synchrony")

In [ ]:
def episode_weight(
    rng: np.random.Generator,
    n_episodes: int = SYNC_N_EPISODES,
    width: tuple[float, float] = SYNC_EPISODE_WIDTH,
    spread: bool = False,
) -> np.ndarray:
    """``(windows,)`` weight of the shared signal: a few brief Gaussian episodes.

    With ``spread``, each episode falls at a random point of its own equal share of
    the track, so no two merge into one peak.
    """
    windows = np.arange(SYNC_N_WINDOWS)
    if spread:
        jitter = rng.uniform(0.25, 0.75, n_episodes)
        centres = 0.1 + 0.8 * (np.arange(n_episodes) + jitter) / n_episodes
    else:
        centres = rng.uniform(0.1, 0.9, n_episodes)
    centres = centres * SYNC_N_WINDOWS
    widths = rng.uniform(*width, n_episodes) * SYNC_N_WINDOWS
    heights = rng.uniform(0.6, 1.0, n_episodes)
    return sum(
        h * np.exp(-0.5 * ((windows - c) / w) ** 2)
        for c, w, h in zip(centres, widths, heights)
    )


def synthetic_embeddings(
    rng: np.random.Generator,
    weight: np.ndarray,
    strength: float = SYNC_EPISODE_STRENGTH,
    smoothing: float = SYNC_SMOOTHING,
) -> np.ndarray:
    """``(subjects, windows, dim)`` embeddings sharing a vector in proportion to
    ``weight``."""
    shape = (SYNC_N_SUBJECTS, SYNC_N_WINDOWS, SYNC_DIM)

    def drift(size: tuple[int, ...]) -> np.ndarray:
        # Smooth along the window axis, so neighbouring windows embed alike.
        noise = lowpass(np.moveaxis(rng.standard_normal(size), -2, -1), 1.0, smoothing)
        noise = np.moveaxis(noise, -1, -2)
        return noise / noise.std()

    shared = drift((SYNC_N_WINDOWS, SYNC_DIM))
    own = drift(shape)
    return own + strength * weight[None, :, None] * shared[None]


def score_synthetic(
    seed: int,
    n_episodes: int = SYNC_N_EPISODES,
    width: tuple[float, float] = SYNC_EPISODE_WIDTH,
    strength: float = SYNC_EPISODE_STRENGTH,
    spread: bool = False,
    smoothing: float = SYNC_SMOOTHING,
) -> tuple[np.ndarray, pd.DataFrame]:
    """Episode weight and the stage-08 scores of one synthetic track."""
    rng = np.random.default_rng(seed)
    weight = episode_weight(rng, n_episodes, width, spread)
    scores = score_track(
        synthetic_embeddings(rng, weight, strength, smoothing),
        starts_s=np.arange(SYNC_N_WINDOWS, dtype=float),
        window_s=1.0,
        n_null=SYNC_N_NULL,
        null_quantile=SYNC_NULL_QUANTILE,
        min_shift=SYNC_N_WINDOWS // 10,
        seed=seed,
    )
    return weight, scores


def _draw_synchrony(ax: plt.Axes, synchrony: np.ndarray, threshold: float) -> None:
    span = synchrony.max() - synchrony.min()
    bottom = synchrony.min() - 0.05 * span
    ax.axhspan(
        bottom, threshold, color=SYNC_BELOW_COLOR, alpha=SYNC_BELOW_ALPHA, linewidth=0
    )
    ax.plot(
        np.linspace(0.0, 1.0, synchrony.size),
        synchrony,
        color=SYNC_COLOR,
        linewidth=SYNC_LINE_WIDTH,
    )
    ax.axhline(
        threshold,
        color=SYNC_THRESHOLD_COLOR,
        linewidth=SYNC_LINE_WIDTH * 0.75,
        linestyle="--",
    )
    ax.set_xlim(0, 1)
    ax.set_ylim(bottom, synchrony.max() + 0.05 * span)
    ax.axis("off")


def plot_synchrony(
    synchrony: np.ndarray,
    threshold: float,
    save_path: str | os.PathLike | None = None,
) -> plt.Figure:
    """Synchrony curve over a grey band marking the sub-threshold range."""
    fig, ax = plt.subplots(figsize=SYNC_FIGSIZE)
    _draw_synchrony(ax, synchrony, threshold)
    fig.tight_layout()
    if save_path is not None:
        fig.savefig(save_path, dpi=300, bbox_inches="tight", transparent=TRANSPARENT)
    return fig

In [ ]:
SYNC_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

for name, seed in SYNC_SEEDS.items():
    _, scores = score_synthetic(seed)
    fig = plot_synchrony(
        scores["synchrony"].to_numpy(),
        scores["null_threshold"].iloc[0],
        save_path=SYNC_OUTPUT_DIR / f"{name}.png",
    )
    fig.savefig(
        SYNC_OUTPUT_DIR / f"{name}.svg", bbox_inches="tight", transparent=TRANSPARENT
    )
    plt.show()
    print(f"{name}: {scores['candidate'].mean():.0%} of windows above threshold")

## Subjects' EEG, and synchrony over it

Two figures from the same synthetic EEG, one row per shown subject (S1, S2, ...),
rows separated by thin lines:

- **Subject EEG** (`subject_eeg/`): each row is the subject label, a small sensor map
  of the 257-channel GSN-HydroCel cap and three EEG traces. No synchrony.
- **Synchrony over EEG** (`synchrony_eeg/`): the synchrony curve on top, the same rows
  below it with labels only. Each synchronous interval (a run of windows above the
  threshold, merged by `candidate_intervals` from stage 08) is a dashed box spanning
  the curve and all rows, and the EEG inside it is drawn in the synchrony colour.
  This figure has its own episode settings (fewer, taller peaks) so the standalone
  synchrony figure above is unchanged.

The shared behaviour is built in, not just highlighted: each subject's EEG is its own
signal blended with one group signal common to all subjects, and the blend follows
the same episode weight that drives the synchrony curve. Inside the boxes every
subject's traces therefore converge on the same waveform; outside they are unrelated.

In [ ]:
SYNC_EEG_SUBJECTS = 3  # rows shown, labelled S1, S2, ...
SYNC_EEG_CHANNELS = 3  # EEG traces per subject
SYNC_EEG_DURATION = 6.0  # seconds of EEG spread over the track; more is denser
SYNC_EEG_SHARING = 1.4  # how far the EEG converges on the shared signal at a peak
SYNC_EEG_ALPHA = 0.6  # alpha-burst strength; big bursts force every trace smaller
SYNC_EEG_LOWPASS_HZ = 45.0  # higher keeps more fast detail: coarser traces
SYNC_EEG_ROUGHNESS = 0.15  # broadband noise on top; 0 for clean traces
SYNC_EEG_MIN_WINDOWS = 6  # boxes only for intervals at least this long
# Synchrony episodes for this figure only: few, tall peaks, spread along the track.
SYNC_EEG_EPISODES = 3
SYNC_EEG_EPISODE_WIDTH = (0.012, 0.02)  # range, as a fraction of the track
SYNC_EEG_EPISODE_STRENGTH = 1.8
SYNC_EEG_SMOOTHING = 0.05  # curve coarseness, as SYNC_SMOOTHING; lower is smoother

SYNC_EEG_WIDTH = 5.5  # inches
SYNC_EEG_CURVE_HEIGHT = 1.8  # inches for the synchrony curve
SYNC_EEG_ROW_HEIGHT = 0.8  # inches per subject row
SYNC_EEG_LABEL_WIDTH = 0.05  # share of the width for the subject labels
SYNC_EEG_CAP_WIDTH = 0.08  # share of the width for the sensor maps
SYNC_EEG_CAP_POINTSIZE = 0.08  # electrode dots on the cap
SYNC_EEG_CAP_LINEWIDTH = 0.6  # head outline
SYNC_EEG_SEPARATOR_COLOR = "#d1d5db"
SYNC_EEG_BOX_ALPHA = 0.08  # fill of the dashed boxes

SUBJECT_EEG_OUTPUT_DIR = ProjectPaths.analysis_plots_dir("conference", "subject_eeg")
SYNC_EEG_OUTPUT_DIR = ProjectPaths.analysis_plots_dir("conference", "synchrony_eeg")

In [ ]:
def subject_eeg(rng: np.random.Generator, weight: np.ndarray) -> np.ndarray:
    """``(subjects, channels, samples)`` EEG converging on one group signal where
    ``weight`` is high."""
    kwargs = {
        "duration": SYNC_EEG_DURATION,
        "alpha": SYNC_EEG_ALPHA,
        "shared": 0.5,
        "lowpass_hz": SYNC_EEG_LOWPASS_HZ,
        "roughness": SYNC_EEG_ROUGHNESS,
    }
    group = synthetic_eeg(rng, SYNC_EEG_CHANNELS, **kwargs)
    own = np.stack(
        [
            synthetic_eeg(rng, SYNC_EEG_CHANNELS, **kwargs)
            for _ in range(SYNC_EEG_SUBJECTS)
        ]
    )
    windows = np.linspace(0, SYNC_N_WINDOWS - 1, own.shape[-1])
    mix = np.interp(windows, np.arange(SYNC_N_WINDOWS), weight)
    mix = np.clip(SYNC_EEG_SHARING * mix, 0.0, 1.0)
    # Constant-variance blend, so a trace does not shrink while it converges.
    return np.sqrt(1 - mix**2) * own + mix * group


def cap_info() -> mne.Info:
    """Sensor positions of the GSN-HydroCel 257 cap, for drawing the sensor maps."""
    montage = mne.channels.make_standard_montage("GSN-HydroCel-257")
    info = mne.create_info(montage.ch_names, SFREQ, "eeg")
    info.set_montage(montage)
    return info


def _subject_scale(eeg: np.ndarray) -> float:
    """One trace scale for all subjects: neighbouring traces never meet, and no
    trace reaches past the edge of its subject's row."""
    scale = _non_overlapping_scale(list(eeg), TRACE_SPACING)
    reach = max(eeg[:, 0].max(), -eeg[:, -1].min())
    return min(scale, (1 - TRACE_MIN_GAP) * 0.5 * TRACE_SPACING / reach)


def _subject_rows_figure(
    n_subjects: int, columns: list[float], curve: bool
) -> tuple[plt.Figure, np.ndarray]:
    """Figure with one row per subject (and a curve row on top if ``curve``), with
    every axis off and separators between the subject rows."""
    heights = [SYNC_EEG_CURVE_HEIGHT] * curve + [SYNC_EEG_ROW_HEIGHT] * n_subjects
    fig, axes = plt.subplots(
        len(heights),
        len(columns),
        figsize=(SYNC_EEG_WIDTH, sum(heights)),
        gridspec_kw={
            "width_ratios": columns,
            "height_ratios": heights,
            "hspace": 0.0,
            "wspace": 0.02,
        },
        squeeze=False,
    )
    fig.subplots_adjust(left=0.01, right=0.99, bottom=0.01, top=0.99)
    for ax in axes.flat:
        ax.axis("off")
    rows = axes[int(curve) :]
    left, right = rows[0, 0].get_position().x0, rows[0, -1].get_position().x1
    for row in rows[1:]:
        y = row[-1].get_position().y1
        fig.add_artist(
            Line2D([left, right], [y, y], color=SYNC_EEG_SEPARATOR_COLOR, linewidth=0.6)
        )
    return fig, axes


def _draw_subject_row(
    ax_label: plt.Axes,
    ax_eeg: plt.Axes,
    subject: int,
    traces: np.ndarray,
    scale: float,
    inside: np.ndarray | None = None,
) -> None:
    """Label and stacked traces of one subject; ``inside`` marks samples to draw in
    the synchrony colour."""
    ax_label.text(
        0.5,
        0.5,
        f"S{subject + 1}",
        ha="center",
        va="center",
        fontsize=LABEL_FONTSIZE,
        color=LINE_COLOR,
        transform=ax_label.transAxes,
    )
    x = np.linspace(0.0, 1.0, traces.shape[-1])
    for i, trace in enumerate(traces):
        y = scale * trace - i * TRACE_SPACING
        ax_eeg.plot(x, y, color=LINE_COLOR, linewidth=LINE_WIDTH * 0.8)
        if inside is not None:
            ax_eeg.plot(
                x,
                np.where(inside, y, np.nan),
                color=SYNC_COLOR,
                linewidth=LINE_WIDTH * 0.8,
            )
    ax_eeg.set_xlim(0, 1)
    ax_eeg.set_ylim(-(len(traces) - 0.5) * TRACE_SPACING, 0.5 * TRACE_SPACING)


def plot_subject_eeg(
    eeg: np.ndarray,
    cap: mne.Info,
    save_path: str | os.PathLike | None = None,
) -> plt.Figure:
    """One row per subject: label, sensor map of the cap and EEG traces."""
    columns = [
        SYNC_EEG_LABEL_WIDTH,
        SYNC_EEG_CAP_WIDTH,
        1 - SYNC_EEG_LABEL_WIDTH - SYNC_EEG_CAP_WIDTH,
    ]
    fig, axes = _subject_rows_figure(len(eeg), columns, curve=False)
    scale = _subject_scale(eeg)
    for subject, (ax_label, ax_cap, ax_eeg) in enumerate(axes):
        mne.viz.plot_sensors(
            cap,
            axes=ax_cap,
            show=False,
            pointsize=SYNC_EEG_CAP_POINTSIZE,
            linewidth=SYNC_EEG_CAP_LINEWIDTH,
        )
        ax_cap.set_title("")
        _draw_subject_row(ax_label, ax_eeg, subject, eeg[subject], scale)
    if save_path is not None:
        fig.savefig(save_path, dpi=300, bbox_inches="tight", transparent=TRANSPARENT)
    return fig


def plot_synchrony_with_eeg(
    synchrony: np.ndarray,
    threshold: float,
    intervals: pd.DataFrame,
    eeg: np.ndarray,
    save_path: str | os.PathLike | None = None,
) -> plt.Figure:
    """Synchrony curve above one labelled row per subject, synchronous intervals
    boxed across all of them."""
    columns = [SYNC_EEG_LABEL_WIDTH, 1 - SYNC_EEG_LABEL_WIDTH]
    fig, axes = _subject_rows_figure(len(eeg), columns, curve=True)
    _draw_synchrony(axes[0, 1], synchrony, threshold)
    # Leave a little air between the curve and the first subject row.
    low, high = axes[0, 1].get_ylim()
    axes[0, 1].set_ylim(low - 0.08 * (high - low), high)

    # Window i is centred at i / (n - 1) on the synchrony axis and covers half a window
    # either side; end_s is one past the interval's last window.
    last = synchrony.size - 1
    spans = [
        (max(0.0, (s - 0.5) / last), min(1.0, (e - 0.5) / last))
        for s, e in zip(intervals.start_s, intervals.end_s)
    ]
    x = np.linspace(0.0, 1.0, eeg.shape[-1])
    inside = np.zeros(x.size, dtype=bool)
    for lo, hi in spans:
        inside |= (x >= lo) & (x <= hi)

    scale = _subject_scale(eeg)
    for subject, (ax_label, ax_eeg) in enumerate(axes[1:]):
        _draw_subject_row(ax_label, ax_eeg, subject, eeg[subject], scale, inside)

    # Boxes span the curve and every subject row: x in data units, y in figure
    # fractions.
    top, bottom = axes[0, 1].get_position().y1, axes[-1, 1].get_position().y0
    transform = blended_transform_factory(axes[1, 1].transData, fig.transFigure)
    for lo, hi in spans:
        for style in (
            {"facecolor": SYNC_COLOR, "alpha": SYNC_EEG_BOX_ALPHA, "linewidth": 0},
            {"fill": False, "edgecolor": SYNC_COLOR, "linestyle": "--", "linewidth": 1},
        ):
            fig.add_artist(
                Rectangle(
                    (lo, bottom), hi - lo, top - bottom, transform=transform, **style
                )
            )
    if save_path is not None:
        fig.savefig(save_path, dpi=300, bbox_inches="tight", transparent=TRANSPARENT)
    return fig

In [ ]:
SUBJECT_EEG_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
SYNC_EEG_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
cap = cap_info()

for name, seed in SYNC_SEEDS.items():
    weight, scores = score_synthetic(
        seed,
        SYNC_EEG_EPISODES,
        SYNC_EEG_EPISODE_WIDTH,
        SYNC_EEG_EPISODE_STRENGTH,
        spread=True,
        smoothing=SYNC_EEG_SMOOTHING,
    )
    intervals = candidate_intervals(scores)
    eeg = subject_eeg(np.random.default_rng(seed + 1), weight)
    figures = {
        SUBJECT_EEG_OUTPUT_DIR: plot_subject_eeg(eeg, cap),
        SYNC_EEG_OUTPUT_DIR: plot_synchrony_with_eeg(
            scores["synchrony"].to_numpy(),
            scores["null_threshold"].iloc[0],
            intervals[intervals.n_windows >= SYNC_EEG_MIN_WINDOWS],
            eeg,
        ),
    }
    for output_dir, fig in figures.items():
        for suffix in ("png", "svg"):
            fig.savefig(
                output_dir / f"{name}.{suffix}",
                dpi=300,
                bbox_inches="tight",
                transparent=TRANSPARENT,
            )
        plt.show()
    n_boxed = (intervals.n_windows >= SYNC_EEG_MIN_WINDOWS).sum()
    print(f"{name}: {n_boxed} of {len(intervals)} synchronous intervals boxed")

## TF maps for the presentation

First component only, saved as bare PDFs at `TF_DPI` and PNGs at `TF_PNG_DPI`: one
file per method and condition holding just the TF map, its onset/offset and 40 Hz guide
lines and the tick numbers — no titles, axis or condition labels, no colour bar.

Every map is **trial-normalised** the way the SNR courses are, at every frequency: each
trial of each frequency minus its own pre-stimulus mean, divided by the pre-stimulus SD
pooled over all trials of that condition, per participant; the median over trials, then
the mean over participants. The unit is then the baseline SD, an SNR, whatever the
source's own scale, so the two methods and the two conditions are comparable directly.
The SD is pooled per condition rather than over both: both methods' sources are
z-scored per condition upstream (the CorrCA inputs, `zscore_mode = per_condition`;
the stage-00 PCA, fitted per recording), and only a per-condition SD cancels that
scale. With `TF_SHARED_SCALE` all four maps share one symmetric colour scale, carried
by `tf_colorbar_baseline_sd.pdf` in baseline-SD units.

- **CorrCA** (`corrca_cc1_<condition>.pdf`): CC1 of the stored stage-06b JoinedTracks
  fit (full montage, `CORRCA_N_PCA`), from `run_corrca_snr_grid.py`'s
  `*__tf_baseline.npz`.
- **Stage 00, ROI → z-score → PCA** (`stage00_<pipeline>_pc1_<condition>.pdf`): PC1 of
  `run_assr_roi_tf_variants.py`'s `2_roi_zscore_pca` pipeline, from the
  `tf_baseline__*` maps of its `--store_tf_maps` file.

Both are computed by `baseline_tf_map` in `src/analysis/assr_trials.py`, and both jobs
need one rerun to write them:
`jobs/metacentrum/00-preprocessing/run_assr_roi_tf_variants.pbs` and
`qsub -v N_PCA=5 jobs/metacentrum/06b-corrca-condition-comparison/run_snr_grid_tracks.pbs`.

Both rows are drawn by the jobs' own `plot_condition_tf_grid` (which draws
`tf_baseline__<pipeline>.png` and `tf_baseline_maps.png`) and only stripped and
recoloured afterwards (`restyle_tf`, `save_bare_panels`): `DIVERGING_CMAP`, guide lines
in `TF_GUIDE_COLOR` instead of the job's `RdBu_r` and green 40 Hz line.

Methods: [`docs/analyses/06b-corrca-condition-comparison/`](../../docs/analyses/06b-corrca-condition-comparison/README.md)
and [`docs/analyses/00-preprocessing/`](../../docs/analyses/00-preprocessing/README.md).

In [ ]:
TF_DPI = 300  # PDF
TF_PNG_DPI = 600  # every TF figure is also saved as a PNG beside its PDF
TF_COMPONENT = 1  # 1-based: CC1 / PC1
TF_TICK_FONTSIZE = 9
TF_COLORBAR_SIZE = (0.3, 2.6)  # inches
TF_GUIDE_COLOR = LINE_COLOR  # onset/offset and 40 Hz lines; the job draws 40 Hz green
TF_SHARED_SCALE = True  # one colour scale over all four maps; False: one per method

CORRCA_VARIANT = IvaVariants.CORRCA_CHANNEL_JOINED_TRACKS
CORRCA_N_PCA = 5
CORRCA_TF_PATH = (
    ProjectPaths.PROJECT_ROOT
    / "results"
    / ExperimentNames.ASSR.value
    / (
        f"assr_snr_grid__{CORRCA_VARIANT.value}__"
        f"{SpectrumTypeVariants.BROADBAND.value}__pca{CORRCA_N_PCA}__tf_baseline.npz"
    )
)

STAGE00_PIPELINE = "2_roi_zscore_pca"
STAGE00_N_COMPONENTS = 3  # the job's --n_components
STAGE00_TF_PATH = (
    ProjectPaths.PROJECT_ROOT
    / "results"
    / ExperimentNames.ASSR.value
    / "assr_roi_tf_variants"
    / MusicTypeVariants.ASSR.value
    / f"tf_maps__pca{STAGE00_N_COMPONENTS}.npz"
)

TF_OUTPUT_DIR = ProjectPaths.analysis_plots_dir("conference", "tf_maps")

# The 2 x 2 grid: rows top to bottom, columns left to right.
TF_GRID_ROWS = ("roi", "corrca")
TF_GRID_COLUMNS = (ConditionVariants.PLACEBO.value, ConditionVariants.PSILOCYBIN.value)
TF_GRID_PANEL = (3.0, 2.4)  # inches per panel
TF_GRID_SPACING = 0.08  # gap between panels, as a fraction of a panel
TF_GRID_LABEL_FONTSIZE = 11  # the shared axis labels
TF_GRID_LABEL_PAD = 3  # points between the tick numbers and the shared labels

# Filled by the figure cell: row -> condition -> its finished TF axes.
tf_panels: dict[str, dict[str, plt.Axes]] = {}

In [ ]:
def save_tf(fig: plt.Figure, path: Path) -> None:
    """Save *fig* to the PDF *path* and to a high-resolution PNG beside it."""
    for target, dpi in ((path, TF_DPI), (path.with_suffix(".png"), TF_PNG_DPI)):
        fig.savefig(target, dpi=dpi, bbox_inches="tight", transparent=TRANSPARENT)


def restyle_tf(fig: plt.Figure) -> None:
    """Redraw every TF map of *fig* in ``DIVERGING_CMAP`` and its guide lines in
    ``TF_GUIDE_COLOR``; data, colour limits and line styles are unchanged."""
    for ax in fig.axes:
        if not ax.images:
            continue
        ax.images[0].set_cmap(DIVERGING_CMAP)
        for line in ax.lines:
            line.set_color(TF_GUIDE_COLOR)


def save_bare_panels(fig: plt.Figure, panels: dict[Path, plt.Axes]) -> None:
    """Save each listed TF axes of *fig* alone: the map and its tick numbers only.

    Titles, the suptitle and every other axes (the other conditions, colour bars) are
    hidden, so a panel keeps the exact image, scale and guide lines it was drawn with.
    """
    fig.suptitle("")
    for ax in panels.values():
        ax.set_title("")
        ax.set_xlabel("")
        ax.set_ylabel("")
        ax.tick_params(labelsize=TF_TICK_FONTSIZE)
    for path, ax in panels.items():
        for other in fig.axes:
            other.set_visible(other is ax)
        save_tf(fig, path)
    for other in fig.axes:
        other.set_visible(True)


def save_colorbar(ax: plt.Axes, path: Path, relative: bool = True) -> float:
    """Save a bare colour bar for the image on *ax*, relative to its symmetric limit
    (-1 to 1) or, with ``relative=False``, in the map's own units; return the limit."""
    image = ax.images[0]
    fig, cax = plt.subplots(figsize=TF_COLORBAR_SIZE)
    # A mappable of its own (the image belongs to another figure). Relative: on -1..1,
    # the map being drawn on ±limit, so the bar's ends are ±limit in its own units.
    norm = Normalize(-1.0, 1.0) if relative else image.norm
    fig.colorbar(
        ScalarMappable(norm=norm, cmap=image.cmap),
        cax=cax,
        ticks=[-1.0, -0.5, 0.0, 0.5, 1.0] if relative else None,
    )
    cax.tick_params(labelsize=TF_TICK_FONTSIZE)
    save_tf(fig, path)
    plt.close(fig)
    return float(image.get_clim()[1])

In [ ]:
TF_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
missing = [str(p) for p in (STAGE00_TF_PATH, CORRCA_TF_PATH) if not p.exists()]
stage00 = None if missing else np.load(STAGE00_TF_PATH)
if stage00 is not None and f"tf_baseline__{STAGE00_PIPELINE}__Placebo" not in stage00:
    missing.append(f"tf_baseline__* in {STAGE00_TF_PATH.name}")
if missing:
    print(f"missing {missing}: rerun the two jobs named above, then this cell.")
else:
    corrca_tf = np.load(CORRCA_TF_PATH)
    epoch_times, freqs = stage00["epoch_times"], stage00["freqs"]
    if not (
        np.allclose(corrca_tf["epoch_times"], epoch_times)
        and np.allclose(corrca_tf["freqs"], freqs)
    ):
        raise ValueError("The two runs' TF maps are on different time/frequency axes.")
    k = TF_COMPONENT - 1
    # Row -> condition -> (1, F, W): the mean over participants of each participant's
    # trial-median map, as the SNR courses are read.
    keys = {
        "roi": (stage00, f"tf_baseline__{STAGE00_PIPELINE}__{{}}"),
        "corrca": (corrca_tf, "tf_baseline__{}"),
    }
    group = {
        row: {c: store[key.format(c)][k : k + 1].mean(axis=1) for c in TF_GRID_COLUMNS}
        for row, (store, key) in keys.items()
    }
    cohorts = {
        "roi": [str(p) for p in stage00["participants"]],
        "corrca": [str(p) for p in corrca_tf["participants"]],
    }
    if sorted(cohorts["roi"]) != sorted(cohorts["corrca"]):
        raise ValueError(f"The two runs' cohorts differ: {cohorts}")
    shared = max(
        float(np.nanmax(np.abs(m))) for by_c in group.values() for m in by_c.values()
    )

    titles = {
        "roi": f"{STAGE00_PIPELINE} PC{TF_COMPONENT}",
        "corrca": f"CorrCA {COMPONENT_PREFIX}{TF_COMPONENT}",
    }
    stems = {
        "roi": f"stage00_{STAGE00_PIPELINE}_pc{TF_COMPONENT}",
        "corrca": f"corrca_cc{TF_COMPONENT}",
    }
    for row in TF_GRID_ROWS:
        fig = plot_condition_tf_grid(
            group[row],
            {c: [""] for c in TF_GRID_COLUMNS},
            list(TF_GRID_COLUMNS),
            epoch_times,
            freqs,
            iva_quality.ASSR_FREQ,
            f"{titles[row]}, trial-normalised (n={len(cohorts[row])})",
            None,
            show_difference=False,
            close=False,
            colorbar_label="baseline SD",
        )
        restyle_tf(fig)
        # The TF axes are the ones holding an image; the rest are their colour bars.
        tf_axes = [ax for ax in fig.axes if ax.images]
        if TF_SHARED_SCALE:
            for ax in tf_axes:
                ax.images[0].set_clim(-shared, shared)
        stem = TF_OUTPUT_DIR / stems[row]
        panels = {
            stem.with_name(f"{stem.name}_{c.lower()}.pdf"): ax
            for c, ax in zip(TF_GRID_COLUMNS, tf_axes)
        }
        save_bare_panels(fig, panels)
        tf_panels[row] = dict(zip(TF_GRID_COLUMNS, tf_axes))
        if not TF_SHARED_SCALE:
            vlim = save_colorbar(
                tf_axes[0], stem.with_name(f"{stem.name}_colorbar.pdf"), relative=False
            )
            print(f"{row}: own scale ±{vlim:.3g} baseline SD")
        plt.show()
        print(f"saved {sorted(p.name for p in panels)}")
    if TF_SHARED_SCALE:
        bar = TF_OUTPUT_DIR / "tf_colorbar_baseline_sd.pdf"
        vlim = save_colorbar(tf_panels["roi"][TF_GRID_COLUMNS[0]], bar, relative=False)
        print(f"one scale for all four maps: ±{vlim:.3g} baseline SD; saved {bar.name}")

### 2 × 2 grid

The four panels above in one figure (`tf_grid_roi_corrca.pdf`): stage-00 ROI → z-score →
PCA on top, CorrCA below; Placebo left, Psilocybin right. No titles; one shared time
and frequency label, tick numbers on the outer edges only, transparent background. Each
panel is copied from its finished figure (image, colour scale, extent and guide lines
unchanged); with `TF_SHARED_SCALE` all four share one scale, the one
`tf_colorbar_baseline_sd.pdf` carries.

In [ ]:
def copy_tf_panel(src: plt.Axes, dst: plt.Axes) -> None:
    """Redraw the TF image and guide lines of *src* on *dst*, unchanged."""
    image = src.images[0]
    dst.imshow(
        image.get_array(),
        aspect="auto",
        origin=image.origin,
        extent=image.get_extent(),
        cmap=image.cmap,
        norm=image.norm,
        interpolation=image.get_interpolation(),
    )
    # axvline / axhline are drawn in blended data-axes coordinates; keep each line in
    # the matching coordinates of the new axes.
    transforms = {
        id(src.get_xaxis_transform()): dst.get_xaxis_transform(),
        id(src.get_yaxis_transform()): dst.get_yaxis_transform(),
    }
    for line in src.lines:
        dst.add_line(
            Line2D(
                *line.get_data(),
                color=line.get_color(),
                linestyle=line.get_linestyle(),
                linewidth=line.get_linewidth(),
                transform=transforms.get(id(line.get_transform()), dst.transData),
            )
        )
    dst.set_xlim(src.get_xlim())
    dst.set_ylim(src.get_ylim())


missing = [row for row in TF_GRID_ROWS if row not in tf_panels]
if missing:
    print(f"no panels for {missing}: run the figure cells above first.")
else:
    width, height = TF_GRID_PANEL
    fig, axes = plt.subplots(
        len(TF_GRID_ROWS),
        len(TF_GRID_COLUMNS),
        figsize=(width * len(TF_GRID_COLUMNS), height * len(TF_GRID_ROWS)),
        gridspec_kw={"wspace": TF_GRID_SPACING, "hspace": TF_GRID_SPACING},
    )
    for r, row in enumerate(TF_GRID_ROWS):
        for col, condition in enumerate(TF_GRID_COLUMNS):
            ax = axes[r, col]
            copy_tf_panel(tf_panels[row][condition], ax)
            ax.tick_params(
                labelsize=TF_TICK_FONTSIZE,
                labelleft=col == 0,
                labelbottom=r == len(TF_GRID_ROWS) - 1,
            )
    # One shared label per axis, centred on the panels and placed just outside the
    # tick numbers (measured once drawn), not at the figure edge.
    fig.canvas.draw()
    renderer = fig.canvas.get_renderer()
    to_figure = fig.transFigure.inverted()
    outer = [
        ax.get_tightbbox(renderer).transformed(to_figure)
        for ax in (*axes[:, 0], *axes[-1, :])
    ]
    pad_x = TF_GRID_LABEL_PAD / 72 / fig.get_figwidth()
    pad_y = TF_GRID_LABEL_PAD / 72 / fig.get_figheight()
    panels_x = (axes[0, 0].get_position().x0 + axes[0, -1].get_position().x1) / 2
    panels_y = (axes[-1, 0].get_position().y0 + axes[0, 0].get_position().y1) / 2
    fig.supxlabel(
        "Time (s)",
        x=panels_x,
        y=min(b.y0 for b in outer) - pad_y,
        va="top",
        fontsize=TF_GRID_LABEL_FONTSIZE,
    )
    fig.supylabel(
        "Frequency (Hz)",
        x=min(b.x0 for b in outer) - pad_x,
        y=panels_y,
        ha="right",
        fontsize=TF_GRID_LABEL_FONTSIZE,
    )
    grid_pdf = TF_OUTPUT_DIR / "tf_grid_roi_corrca.pdf"
    save_tf(fig, grid_pdf)
    plt.show()
    print(f"saved {grid_pdf}")

## 40 Hz SNR over the trial: classical approach vs CorrCA

One panel per condition (Placebo, Psilocybin), each overlaying the two methods' 40 Hz
trial course: the classical stage-00 **ROI PCA PC1** (`2_roi_zscore_pca`, the TF-map
row above) and our **CorrCA CC1** (the `full` rows of the stage-06b SNR grid on the
JoinedTracks fit). Drawn with the jobs' own `plot_courses`, the methods passed as the
overlaid series, and saved as a PDF and PNG with no title.

Every course is in the same units, an SNR over time: each trial referenced to its own
pre-stimulus mean and divided by the pooled baseline SD, the median over trials per
participant, then the mean ± SEM over participants. The two methods are on the same
participants, which the cell checks. The stimulus is shaded grey and `COURSE_WINDOW`
gold. There is no p-value box: the jobs test Placebo against Psilocybin, not one
method against the other.

`COURSE_CLASSICAL` picks the classical row; set it to `"ROI"` for the plain binary ROI
(06b's `ASSR-mask (full)`, the electrode average) instead of its first PC.

Methods: [`docs/analyses/06b-corrca-condition-comparison/`](../../docs/analyses/06b-corrca-condition-comparison/README.md).

In [ ]:
COURSE_SELECTION = "40hz"  # or "35-45hz"
COURSE_WINDOW = (0.0, 0.5)  # shaded gold; or (0.2, 0.5)
COURSE_CONDITIONS = TF_GRID_COLUMNS  # one panel each, left to right
COURSE_CLASSICAL = "ROI PCA PC1"  # or "ROI"
COURSE_YLABEL = "40 Hz SNR (baseline SD)"

_RESULTS = ProjectPaths.PROJECT_ROOT / "results" / ExperimentNames.ASSR.value
_CORRCA_GRID = (
    f"assr_snr_grid__{CORRCA_VARIANT.value}__{SpectrumTypeVariants.BROADBAND.value}"
    f"__pca{CORRCA_N_PCA}"
)
_STAGE00_COURSES = (
    STAGE00_TF_PATH.parent / f"snr_courses__full__pca{STAGE00_N_COMPONENTS}.npz"
)

# Method label -> the courses file, the run within it, and the source within that run.
COURSE_SOURCES = {
    "ROI PCA PC1": (_STAGE00_COURSES, STAGE00_PIPELINE, f"PC {TF_COMPONENT}"),
    "ROI": (_STAGE00_COURSES, STAGE00_PIPELINE, "ASSR-mask (full)"),
    f"CorrCA {COMPONENT_PREFIX}{TF_COMPONENT}": (
        _RESULTS / f"{_CORRCA_GRID}__courses.npz",
        "full",
        f"{COMPONENT_PREFIX} {TF_COMPONENT}",
    ),
}
# Classical first, then ours; near-black and the Okabe-Ito bluish green, which stay
# apart under every colour-vision deficiency and differ from the condition colours.
COURSE_METHODS = (COURSE_CLASSICAL, f"CorrCA {COMPONENT_PREFIX}{TF_COMPONENT}")
COURSE_COLORS = dict(zip(COURSE_METHODS, (LINE_COLOR, "#009E73")))

COURSE_OUTPUT_DIR = ProjectPaths.analysis_plots_dir("conference", "trial_courses")

In [ ]:
missing = [
    str(COURSE_SOURCES[m][0])
    for m in COURSE_METHODS
    if not COURSE_SOURCES[m][0].exists()
]
if missing:
    print(f"missing {missing}: run the two jobs named above.")
else:
    loaded = {m: load_snr_courses(COURSE_SOURCES[m][0]) for m in COURSE_METHODS}
    cohorts = {m: sorted(item[0], key=int) for m, item in loaded.items()}
    participants = cohorts[COURSE_METHODS[0]]
    if any(cohort != participants for cohort in cohorts.values()):
        raise ValueError(f"The methods' cohorts differ: {cohorts}")
    times = loaded[COURSE_METHODS[0]][2]
    if any(not np.allclose(item[2], times) for item in loaded.values()):
        raise ValueError("The methods' epoch time axes differ.")

    # Method -> (participants, conditions, times): the conditions take the place of
    # plot_courses's sources, one panel each, and the methods that of its conditions.
    course = {}
    for m, (run_participants, sources, _, courses) in loaded.items():
        _, run, source = COURSE_SOURCES[m]
        order = [run_participants.index(p) for p in participants]
        by_condition = courses[(run, COURSE_SELECTION)]
        course[m] = np.stack(
            [by_condition[c][order, sources.index(source)] for c in COURSE_CONDITIONS],
            axis=1,
        )

    fig = plot_courses(
        course,
        times,
        COURSE_WINDOW,
        None,
        list(COURSE_CONDITIONS),
        list(COURSE_METHODS),
        0.05,
        "",
        None,
        close=False,
        colors=COURSE_COLORS,
    )
    fig.axes[0].set_ylabel(COURSE_YLABEL)
    # The job's lower-right legend covers the post-stimulus courses here.
    fig.axes[0].legend(loc="upper right", frameon=False, fontsize=9)
    COURSE_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    lo, hi = COURSE_WINDOW
    slug = COURSE_CLASSICAL.lower().replace(" ", "_")
    course_pdf = COURSE_OUTPUT_DIR / (
        f"{slug}_vs_corrca_cc{TF_COMPONENT}__{COURSE_SELECTION}__{lo:g}-{hi:g}s.pdf"
    )
    save_tf(fig, course_pdf)
    plt.show()
    print(f"saved {course_pdf} (n={len(participants)})")

## What the condition tests compare: four methods

The Placebo − Psilocybin test every read-out runs, shown as what it is run on: one
value per participant and condition, the median over trials of the 40 Hz SNR averaged
over `STATS_WINDOW` (each trial referenced to its own pre-stimulus window, in baseline
SD). First component of each method, in this order:

- **ICA IC1**: stage 07, joint ICA on JoinedTracks, `full` rows.
- **IVA IC1**: stage 06, `iva_channel_joined_tracks`, `full` rows.
- **CorrCA CC1**: stage 06b, JoinedTracks fit, `full` rows.
- **ROI PCA PC1** — the **baseline**: stage 00, `2_roi_zscore_pca`, the classical
  approach the three learned methods are compared against.

One figure holds every method as a panel on a shared y-axis, with no x-axis (the
condition legend on top names the bars). Bars are the mean ± SEM over participants and
the grey lines join each participant's two values; the p-value is the job's own (exact
Wilcoxon signed-rank on the paired differences, one-sided Placebo > Psilocybin,
uncorrected), read from its CSV. The summary figure then puts every method's
Psilocybin − Placebo difference side by side (mean ± SEM, a dot per participant). In
both, the baseline is set apart by hatched bars, and in the summary by its colour:
near-black against the learned methods' green, as in the trial-course figure.

The cell checks that each values file reproduces its CSV's test median, so the bars
are exactly what was tested. The values come from the files the jobs write beside
their CSVs (`*values.npz`); stage 00 and 06b have them, and IVA and ICA need one rerun:
`jobs/metacentrum/06-iva-condition-comparison/run_snr_grid_tracks.pbs` and
`jobs/metacentrum/07-ica-condition-comparison/run_jica_tracks.pbs`. A method without
its file is skipped.

In [ ]:
STATS_SELECTION = "40hz"  # or "35-45hz"
STATS_WINDOW = (0.0, 0.5)  # s; ICA has (0, 0.5) unless its job set another window
STATS_CONDITIONS = TF_GRID_COLUMNS
STATS_YLABEL = "40 Hz SNR (baseline SD)"
STATS_PANEL_SIZE = (1.7, 3.0)  # inches per method panel
STATS_SUMMARY_FIGSIZE = (5.6, 3.0)  # inches
STATS_OURS_COLOR = "#009E73"  # summary bars of the learned methods
STATS_BASELINE_COLOR = LINE_COLOR  # summary bar of the baseline
STATS_BASELINE_HATCH = "///"  # sets the baseline's bars apart in both figures
STATS_PAIR_COLOR = "#9ca3af"  # participant lines and dots
STATS_IVA_N_PCA = 5
STATS_JICA_N_ICA = 5

_lo, _hi = STATS_WINDOW
_GRID_WINDOW = f"{_lo:g}-{_hi:g}s"  # stage 00 / 06 / 06b
# run_wavelet_jica.py labels its one window by how it was set.
_JICA_WINDOW = (
    f"0-{_hi:.3f} s (paradigm default)"
    if STATS_WINDOW == (0.0, AssrEpoch.STIMULUS_DURATION_S)
    else f"{_lo:.3f}-{_hi:.3f} s (custom)"
)
_IVA_GRID = (
    f"assr_snr_grid__iva_channel_joined_tracks__"
    f"{SpectrumTypeVariants.BROADBAND.value}__pca{STATS_IVA_N_PCA}"
)
_JICA_TESTS = f"jica_tests__joined_tracks__ASSR__ica{STATS_JICA_N_ICA}"
_STAGE00_GRID = f"full__pca{STAGE00_N_COMPONENTS}"

# Method -> its test CSV and values file, the CSV column naming the run and the run,
# the window as that run labels it, the source, and whether it is the baseline. The
# order is the figures' left to right.
STATS_METHODS = {
    f"ICA IC{TF_COMPONENT}": {
        "csv": _RESULTS / f"{_JICA_TESTS}.csv",
        "values": _RESULTS / f"{_JICA_TESTS}__values.npz",
        "run_column": "variant",
        "run": "full",
        "window": _JICA_WINDOW,
        "source": f"IC {TF_COMPONENT}",
        "baseline": False,
    },
    f"IVA IC{TF_COMPONENT}": {
        "csv": _RESULTS / f"{_IVA_GRID}.csv",
        "values": _RESULTS / f"{_IVA_GRID}__values.npz",
        "run_column": "variant",
        "run": "full",
        "window": _GRID_WINDOW,
        "source": f"IC {TF_COMPONENT}",
        "baseline": False,
    },
    f"CorrCA {COMPONENT_PREFIX}{TF_COMPONENT}": {
        "csv": _RESULTS / f"{_CORRCA_GRID}.csv",
        "values": _RESULTS / f"{_CORRCA_GRID}__values.npz",
        "run_column": "variant",
        "run": "full",
        "window": _GRID_WINDOW,
        "source": f"{COMPONENT_PREFIX} {TF_COMPONENT}",
        "baseline": False,
    },
    f"ROI PCA PC{TF_COMPONENT}": {
        "csv": STAGE00_TF_PATH.parent / f"snr_grid__{_STAGE00_GRID}.csv",
        "values": STAGE00_TF_PATH.parent / f"snr_values__{_STAGE00_GRID}.npz",
        "run_column": "pipeline",
        "run": STAGE00_PIPELINE,
        "window": _GRID_WINDOW,
        "source": f"PC {TF_COMPONENT}",
        "baseline": True,
    },
}

STATS_OUTPUT_DIR = ProjectPaths.analysis_plots_dir("conference", "condition_tests")

In [ ]:
def method_values(label: str) -> tuple[dict[str, np.ndarray], pd.Series]:
    """One method's per-participant values (condition -> ``(P,)``) and the job's
    Placebo - Psilocybin test row; checks the values reproduce the test's median."""
    method = STATS_METHODS[label]
    _, sources, values = load_snr_values(method["values"])
    cell = values[(method["run"], STATS_SELECTION, method["window"])]
    column = sources.index(method["source"])
    by_condition = {c: cell[c][:, column] for c in STATS_CONDITIONS}

    frame = pd.read_csv(method["csv"])
    test = frame[
        (frame[method["run_column"]] == method["run"])
        & (frame["selection"] == STATS_SELECTION)
        & (frame["window"] == method["window"])
        & (frame["family"] == "contrast")
        & (frame["source"] == method["source"])
    ]
    if len(test) != 1:
        raise ValueError(
            f"{method['csv'].name}: {len(test)} contrast rows for {label}."
        )
    test = test.iloc[0]
    first, second = STATS_CONDITIONS
    median = np.nanmedian(by_condition[first] - by_condition[second])
    if not np.isclose(median, test["median"], rtol=1e-4, atol=1e-6):
        raise ValueError(
            f"{label}: the values give median {median:.4g}, the CSV {test['median']:.4g}"
            " — the two files are from different runs."
        )
    return by_condition, test


def _mean_sem(values: np.ndarray) -> tuple[float, float]:
    finite = values[np.isfinite(values)]
    return float(finite.mean()), float(finite.std(ddof=1) / np.sqrt(finite.size))


def _hatch(method: str) -> str | None:
    return STATS_BASELINE_HATCH if STATS_METHODS[method]["baseline"] else None


def plot_condition_bars(
    loaded: dict[str, tuple[dict[str, np.ndarray], pd.Series]],
) -> plt.Figure:
    """Every method as a panel on one y-axis: Placebo vs Psilocybin bars (mean +/-
    SEM), each participant's pair joined, the job's p-value over a bracket. No x-axis;
    a condition legend on top names the bars, and the baseline's bars are hatched."""
    width, height = STATS_PANEL_SIZE
    fig, axes = plt.subplots(
        1,
        len(loaded),
        figsize=(width * len(loaded), height),
        sharey=True,
        squeeze=False,
    )
    xs = np.arange(len(STATS_CONDITIONS))
    every = np.concatenate([v for by_c, _ in loaded.values() for v in by_c.values()])
    low, high = np.nanmin(every), np.nanmax(every)
    span = high - low
    for ax, (label, (by_condition, test)) in zip(axes[0], loaded.items()):
        paired = np.column_stack([by_condition[c] for c in STATS_CONDITIONS])  # (P, 2)
        for row in paired:
            ax.plot(xs, row, color=STATS_PAIR_COLOR, lw=0.8, zorder=2)
            ax.scatter(xs, row, s=10, color=STATS_PAIR_COLOR, zorder=3, linewidths=0)
        for x, c in zip(xs, STATS_CONDITIONS):
            mean, sem = _mean_sem(by_condition[c])
            ax.bar(
                x,
                mean,
                width=0.6,
                color=CONDITION_COLORS[c],
                alpha=0.85,
                hatch=_hatch(label),
                edgecolor="white",
                linewidth=0,
                zorder=1,
            )
            ax.errorbar(
                x, mean, yerr=sem, color=LINE_COLOR, capsize=4, lw=1.2, zorder=4
            )
        top = np.nanmax(paired) + 0.06 * span
        ax.plot(xs, [top, top], color=LINE_COLOR, lw=0.9)
        ax.text(
            xs.mean(),
            top + 0.015 * span,
            f"p = {test['p']:.3f}",
            ha="center",
            va="bottom",
            fontsize=9,
            color=LINE_COLOR,
        )
        ax.axhline(0.0, color=LINE_COLOR, lw=0.8)
        baseline = STATS_METHODS[label]["baseline"]
        ax.set_title(f"{label}\n(baseline)" if baseline else label, fontsize=10)
        ax.set_xticks([])
        ax.spines[["top", "right", "bottom"]].set_visible(False)
        if ax is not axes[0, 0]:
            ax.spines["left"].set_visible(False)
            ax.tick_params(left=False)
    axes[0, 0].set_ylim(min(0.0, low) - 0.05 * span, high + 0.2 * span)
    axes[0, 0].set_ylabel(STATS_YLABEL)
    handles = [
        Rectangle((0, 0), 1, 1, color=CONDITION_COLORS[c], alpha=0.85, label=c)
        for c in STATS_CONDITIONS
    ]
    fig.tight_layout()
    # Just above the panel titles, measured once drawn.
    fig.canvas.draw()
    renderer = fig.canvas.get_renderer()
    to_figure = fig.transFigure.inverted()
    titles_top = max(
        ax.title.get_window_extent(renderer).transformed(to_figure).y1 for ax in axes[0]
    )
    fig.legend(
        handles=handles,
        loc="lower center",
        bbox_to_anchor=(0.5, titles_top),
        ncol=len(handles),
        frameon=False,
    )
    return fig


def plot_difference_summary(
    differences: dict[str, np.ndarray], tests: dict[str, pd.Series]
) -> plt.Figure:
    """One bar per method: Psilocybin - Placebo, mean +/- SEM, a dot per participant,
    and the job's p-value above; the baseline in its own colour, hatched."""
    fig, ax = plt.subplots(figsize=STATS_SUMMARY_FIGSIZE)
    xs = np.arange(len(differences))
    rng = np.random.default_rng(0)
    for x, (label, diff) in zip(xs, differences.items()):
        mean, sem = _mean_sem(diff)
        baseline = STATS_METHODS[label]["baseline"]
        ax.bar(
            x,
            mean,
            width=0.6,
            color=STATS_BASELINE_COLOR if baseline else STATS_OURS_COLOR,
            alpha=0.85,
            hatch=_hatch(label),
            edgecolor="white",
            linewidth=0,
            zorder=1,
        )
        ax.errorbar(x, mean, yerr=sem, color=LINE_COLOR, capsize=4, lw=1.2, zorder=4)
        jitter = rng.uniform(-0.12, 0.12, diff.size)
        ax.scatter(
            x + jitter, diff, s=10, color=STATS_PAIR_COLOR, zorder=3, linewidths=0
        )
        ax.text(
            x,
            1.0,
            f"p = {tests[label]['p']:.3f}",
            transform=ax.get_xaxis_transform(),
            ha="center",
            va="bottom",
            fontsize=8.5,
            color=LINE_COLOR,
        )
    ax.axhline(0.0, color=LINE_COLOR, lw=0.8)
    ax.set_xticks(
        xs,
        [
            f"{label}\n(baseline)" if STATS_METHODS[label]["baseline"] else label
            for label in differences
        ],
    )
    second, first = STATS_CONDITIONS[1], STATS_CONDITIONS[0]
    ax.set_ylabel(f"{second} − {first}\n{STATS_YLABEL}")
    ax.spines[["top", "right"]].set_visible(False)
    fig.tight_layout()
    return fig

In [ ]:
missing = {
    label: m["values"]
    for label, m in STATS_METHODS.items()
    if not (m["values"].exists() and m["csv"].exists())
}
for label, path in missing.items():
    print(f"{label}: no {path} yet; rerun its job (named above). Skipped.")
loaded = {
    label: method_values(label) for label in STATS_METHODS if label not in missing
}

if loaded:
    STATS_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    tag = f"{STATS_SELECTION}__{_GRID_WINDOW}"
    fig = plot_condition_bars(loaded)
    path = STATS_OUTPUT_DIR / f"condition_bars__{tag}.pdf"
    save_tf(fig, path)
    plt.show()
    for label, (by_condition, test) in loaded.items():
        n = np.isfinite(by_condition[STATS_CONDITIONS[0]]).sum()
        print(f"{label}: p = {test['p']:.3f}, n = {n}")
    print(f"saved {path}")

    first, second = STATS_CONDITIONS
    fig = plot_difference_summary(
        {label: by_c[second] - by_c[first] for label, (by_c, _) in loaded.items()},
        {label: test for label, (_, test) in loaded.items()},
    )
    path = (
        STATS_OUTPUT_DIR / f"summary_{second.lower()}_minus_{first.lower()}__{tag}.pdf"
    )
    save_tf(fig, path)
    plt.show()
    print(f"saved {path}")

## Topomaps: CorrCA CC1 and the binary ASSR ROI

Two scalp maps, drawn the same way and saved as bare PDFs and PNGs (no titles):

- **CorrCA CC1** (`corrca_cc1.pdf`): the shared forward pattern of the stored
  stage-06b JoinedTracks fit (`CORRCA_N_PCA`), drawn by the job's own
  `plot_shared_topomaps` on its symmetric scale. A separate `corrca_cc1_colorbar.pdf`
  carries that scale as a relative −1 to 1, and the run prints the absolute limit.
- **Binary ROI** (`assr_roi_mask.pdf`): the ASSR electrodes of `config/assr_electrodes/`
  (06b's `ASSR-mask (full)`) as a 0/1 pattern on the same channels, through the same
  function: 1 for a ROI electrode, 0 for the rest, on the same ±1 scale. It is drawn
  with nearest-electrode interpolation and no contours (`TOPO_MASK_INTERP`,
  `TOPO_MASK_CONTOURS`), so it keeps its two values: the job's cubic interpolation
  overshoots a 0/1 step into ripples and a faint negative halo.

Both are drawn in `DIVERGING_CMAP`, the TF maps' scale, so the ROI reads in the same
warm colour as the positive end of CC1. The sign of CC1 is
the one fixed at fit time against these electrodes, so a positive pattern over the ROI
is the expected picture. CC1 is in fact positive on every channel (a whole-head 40 Hz
component, strongest over both temporal regions), so on the symmetric scale the whole
head is warm and its spatial structure sits in the upper half of the scale.

In [ ]:
TOPO_COORDINATES = CoordinateSystems.HYDROGEL_257_NO_FIDUCIALS
# The binary ROI: nearest-electrode interpolation and no contours keep it two-valued;
# "linear" with TOPO_MASK_CONTOURS = 1 gives a soft edge outlined at 0.5 instead.
TOPO_MASK_INTERP = "nearest"
TOPO_MASK_CONTOURS = 0
TOPO_OUTPUT_DIR = ProjectPaths.analysis_plots_dir("conference", "topomaps")

In [ ]:
TOPO_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
corrca_topo = load_iva_components(
    experiment=ExperimentNames.ASSR,
    condition=ConditionVariants.JOINED_TRACKS,
    variant=CORRCA_VARIANT,
    music_type=MusicTypeVariants.ASSR,
    n_pca=CORRCA_N_PCA,
)
topo_info = corrca_topo.topo_info(TOPO_COORDINATES)
k = TF_COMPONENT - 1
roi_mask = assr_electrode_mask(
    list(corrca_topo.channel_names), TOPO_COORDINATES, strict=True
)
# Output stem -> ((1, C) pattern, (1,) ISC for the job's title, stripped below, and
# the drawing options; CC1 keeps the job's own).
topomaps = {
    f"corrca_cc{TF_COMPONENT}": (
        corrca_topo.extras["corrca_shared_patterns"][k : k + 1],
        corrca_topo.extras["corrca_fit_isc"][k : k + 1],
        {},
    ),
    "assr_roi_mask": (
        roi_mask[None].astype(float),
        np.full(1, np.nan),
        {"image_interp": TOPO_MASK_INTERP, "contours": TOPO_MASK_CONTOURS},
    ),
}
for stem, (pattern, isc, options) in topomaps.items():
    fig = plot_shared_topomaps(
        pattern,
        isc,
        topo_info,
        corrca_topo.n_channels,
        [0],
        label=stem,
        cmap=DIVERGING_CMAP,
        **options,
    )
    fig.suptitle("")
    ax = fig.axes[0]
    ax.set_title("")
    save_tf(fig, TOPO_OUTPUT_DIR / f"{stem}.pdf")
    plt.show()
    if stem.startswith("corrca"):
        vlim = save_colorbar(ax, TOPO_OUTPUT_DIR / f"{stem}_colorbar.pdf")
        print(f"{stem}: saved with its colour bar (±{vlim:.3g})")
    else:
        print(f"{stem}: {int(roi_mask.sum())} of {roi_mask.size} electrodes; saved")

## Head (EEG cap)

The sensor map of the 257-channel GSN-HydroCel cap on its own — the head drawn in the
subject EEG figure, larger and with no labels — as a PDF.

In [ ]:
HEAD_SIZE = 3.0  # inches, square
HEAD_POINTSIZE = 2.0  # electrode dots
HEAD_LINEWIDTH = 1.5  # head outline

HEAD_OUTPUT_DIR = ProjectPaths.analysis_plots_dir("conference", "head")
HEAD_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

fig, ax = plt.subplots(figsize=(HEAD_SIZE, HEAD_SIZE))
mne.viz.plot_sensors(
    cap_info(),
    axes=ax,
    show=False,
    pointsize=HEAD_POINTSIZE,
    linewidth=HEAD_LINEWIDTH,
)
ax.set_title("")
ax.axis("off")
head_pdf = HEAD_OUTPUT_DIR / "gsn_hydrocel_257.pdf"
fig.savefig(head_pdf, dpi=TF_DPI, bbox_inches="tight", transparent=TRANSPARENT)
plt.show()
print(f"saved {head_pdf}")